# Road damage detection: train YOLOv8m in the cloud

Trains YOLOv8m on RDD2022 (classes D00, D10, D20, D40), then evaluates it on the val
and test splits. The notebook is self-contained: it writes the project scripts itself,
so you don't need to clone the repository. It was generated by
`scripts/build_cloud_notebooks.py`; edit the scripts and rebuild instead of editing
the code cells by hand.

**Kaggle**
1. *Create → New Notebook → File → Import Notebook*, then upload this file.
2. *Add Input*: the RDD2022 YOLO dataset (`train/ val/ test/` with `images/` and `labels/`),
   or your own fixed `data/` folder uploaded as a private dataset.
3. *Settings*: Accelerator **GPU T4 x2** or **P100**, Internet **on**.
4. *Save Version → Save & Run All (Commit)*. This runs for up to 12 h and keeps `/kaggle/working`.
5. If training stopped at `MAX_HOURS`, create a new version, *Add Input → Your Work →* the
   previous version's output, set `RESUME_FROM` to that path, and commit again. Training
   continues from the last epoch.

**Colab**
1. Upload the notebook, then *Runtime → Change runtime type → GPU*.
2. Put the dataset (folder or `.zip`) on Google Drive and set `DATA_DIR`.
3. Set `DRIVE_DIR` so runs are saved to Drive. After a disconnect, *Run all* again and
   training resumes automatically.

The data is fixed on the fly by `fix_prepared_data.py`. Class 4 (pothole) is remapped
to D40, "other corruption" boxes are dropped, and images are split by 50-frame blocks
so neighbouring frames don't leak between splits. Data that was already fixed is used as-is.

## 1. Settings

In [ ]:
# ---- settings: edit these ----
MODEL_KEY = "yolov8m"     # run name prefix; runs are named <MODEL_KEY>_<IMGSZ>_s<SEED>
WEIGHTS = "yolov8m.pt"   # COCO-pretrained, downloaded automatically
EPOCHS = 30               # keep the same for every model you compare
IMGSZ = 640
BATCH = 16               # for a 16 GB GPU; lower it if CUDA runs out of memory
SEED = 42
DEVICE = "0"              # "cpu" to test without a GPU; "0,1" uses both Kaggle T4s
MAX_HOURS = 10.5          # stop cleanly before the session limit, then run again to resume
DATA_DIR = None           # folder with train/ val/ test/ (or a .zip of it); None = search inputs
DRIVE_DIR = None          # Colab: e.g. "/content/drive/MyDrive/road-damage" keeps runs on Drive
RESUME_FROM = None        # Kaggle: previous version's output added as input, to continue training
EXTRA_ARGS = ""           # extra training flags, e.g. "--set fraction=0.05" for a quick test

## 2. Environment

In [ ]:
import glob, json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB and (DRIVE_DIR or str(DATA_DIR or "").startswith("/content/drive")):
    from google.colab import drive
    drive.mount("/content/drive")

if ON_KAGGLE:   # code + data in /tmp, results in /kaggle/working (saved with the version)
    WORK, OUT = Path("/tmp/rdd"), Path("/kaggle/working")
elif ON_COLAB:
    WORK, OUT = Path("/content/rdd"), Path(DRIVE_DIR or "/content/outputs")
else:
    WORK = Path.cwd() / "cloud_work"
    OUT = Path(DRIVE_DIR) if DRIVE_DIR else WORK / "outputs"
RUNS = OUT / "runs"
for d in (WORK / "scripts", WORK / "configs", RUNS):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
RUN_NAME = f"{MODEL_KEY}_{IMGSZ}_s{SEED}"
WORKERS = min(8, os.cpu_count() or 2)
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} | code+data: {WORK} | outputs: {OUT} | run: {RUN_NAME}")


def run(cmd):
    """Run a command, streaming its output into the notebook; raise if it fails."""
    print("$", " ".join(shlex.quote(str(c)) for c in cmd), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    while chunk := proc.stdout.read1(4096):
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
    if proc.wait():
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")

In [ ]:
%pip install -q "ultralytics>=8.3" pyyaml tqdm pandas matplotlib

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.0f} GB")
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Kaggle: Settings → Accelerator → GPU. Colab: Runtime → Change runtime type → GPU.")

## 3. Project code

These cells are copies of the repository's `scripts/` and `configs/` files.

In [ ]:
%%writefile scripts/utils.py
"""Shared helpers for the road damage detection scripts.

Not meant to be run directly. The other scripts in this folder import from it.
"""
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import yaml

PROJECT_ROOT = Path(__file__).resolve().parents[1]
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
DEFAULT_THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)


# --------------------------------------------------------------------------- #
# General helpers
# --------------------------------------------------------------------------- #
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def save_json(obj, path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=float)


def resolve_weights(weights) -> str:
    """Use the path as given, or relative to the project root. Otherwise return it
    unchanged (so names like 'yolov8s.pt' can be auto-downloaded by Ultralytics)."""
    p = Path(weights)
    if p.exists():
        return str(p)
    if (PROJECT_ROOT / p).exists():
        return str(PROJECT_ROOT / p)
    return str(weights)


def _is_rtdetr(weights: str) -> bool:
    if "rtdetr" in weights.lower() or "rt-detr" in weights.lower():
        return True
    p = Path(weights)
    if p.suffix == ".pt" and p.exists():
        try:
            import torch

            ckpt = torch.load(str(p), map_location="cpu", weights_only=False)
            m = ckpt.get("ema") or ckpt.get("model")
            return type(m).__name__ == "RTDETRDetectionModel"
        except Exception:
            return False
    return False


def load_model(weights, task: str | None = "detect"):
    """Load a YOLO or RT-DETR model. RT-DETR needs its own class because it uses a
    different predictor, so it is detected from the file name or the checkpoint."""
    from ultralytics import RTDETR, YOLO

    weights = resolve_weights(weights)
    if _is_rtdetr(weights):
        return RTDETR(weights)
    return YOLO(weights, task=task) if task else YOLO(weights)


# --------------------------------------------------------------------------- #
# Dataset yaml helpers
# --------------------------------------------------------------------------- #
def resolve_data_yaml(data) -> Path:
    """Write a copy of the dataset yaml with an absolute `path:`.

    Ultralytics resolves relative dataset paths against its own datasets directory,
    not the current folder, so we always hand it an absolute path.
    """
    data_path = Path(data)
    if not data_path.exists():
        data_path = PROJECT_ROOT / data
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset yaml not found: {data}")
    with open(data_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    root = Path(cfg.get("path", data_path.parent))
    if not root.is_absolute():
        root = (PROJECT_ROOT / root).resolve()
    cfg["path"] = root.as_posix()
    out_dir = PROJECT_ROOT / "runs" / "_resolved"
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / data_path.name
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out


def load_names(data_yaml) -> dict[int, str]:
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    names = cfg["names"]
    if isinstance(names, list):
        return {i: n for i, n in enumerate(names)}
    return {int(k): v for k, v in names.items()}


def get_split_dirs(data_yaml, split: str) -> tuple[Path, Path]:
    """Return (images_dir, labels_dir) for a split of a resolved dataset yaml."""
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if split not in cfg:
        raise KeyError(f"Split '{split}' is not defined in {data_yaml}")
    rel = cfg[split]
    if isinstance(rel, list):
        rel = rel[0]
    img_dir = Path(cfg["path"]) / rel
    parts = list(img_dir.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            break
    else:
        raise ValueError(f"Cannot derive the labels folder from {img_dir}")
    return img_dir, Path(*parts)


def list_images(folder) -> list[Path]:
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS)


def read_yolo_labels(path) -> tuple[np.ndarray, np.ndarray]:
    """Read a YOLO label file. Returns (class_ids[n], boxes_xyxy_normalised[n, 4])."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros(0, dtype=int), np.zeros((0, 4))
    arr = np.loadtxt(path, ndmin=2)
    cls = arr[:, 0].astype(int)
    cx, cy, w, h = arr[:, 1], arr[:, 2], arr[:, 3], arr[:, 4]
    xyxy = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)
    return cls, xyxy


# --------------------------------------------------------------------------- #
# Detection metrics (F1 at IoU 0.5, the RDD challenge metric)
# --------------------------------------------------------------------------- #
def box_iou_np(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Pairwise IoU between boxes a[n, 4] and b[m, 4] in xyxy format."""
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.maximum(union, 1e-12)


class DetectionEvaluator:
    """Accumulates predictions over a dataset and computes P / R / F1 at any
    confidence threshold.

    Predictions are matched to ground truth once (greedy, by descending confidence,
    same class, IoU >= iou_thr). Because greedy matching only depends on higher
    scoring boxes, the result for any threshold is just a filter on that single
    matching, so sweeping thresholds is cheap.

    IoU is computed on normalised coordinates, which gives the same value as pixel
    coordinates (IoU is invariant to scaling each axis).
    """

    def __init__(self, nc: int, iou_thr: float = 0.5):
        self.nc = nc
        self.iou_thr = iou_thr
        self.n_gt = np.zeros(nc, dtype=int)
        self._scores, self._cls, self._tp = [], [], []
        self._cache = None

    def add(self, pred_boxes, pred_scores, pred_cls, gt_boxes, gt_cls) -> None:
        self._cache = None
        pred_boxes = np.asarray(pred_boxes, dtype=float).reshape(-1, 4)
        pred_scores = np.asarray(pred_scores, dtype=float).reshape(-1)
        pred_cls = np.asarray(pred_cls).astype(int).reshape(-1)
        gt_boxes = np.asarray(gt_boxes, dtype=float).reshape(-1, 4)
        gt_cls = np.asarray(gt_cls).astype(int).reshape(-1)

        for c in range(self.nc):
            g = gt_boxes[gt_cls == c]
            self.n_gt[c] += len(g)
            m = pred_cls == c
            if not m.any():
                continue
            b, s = pred_boxes[m], pred_scores[m]
            order = np.argsort(-s)
            b, s = b[order], s[order]
            tp = np.zeros(len(b), dtype=bool)
            if len(g):
                ious = box_iou_np(b, g)
                matched = np.zeros(len(g), dtype=bool)
                for i in range(len(b)):
                    cand = np.where(matched, -1.0, ious[i])
                    j = int(cand.argmax())
                    if cand[j] >= self.iou_thr:
                        matched[j] = True
                        tp[i] = True
                        if matched.all():
                            break
            self._scores.append(s)
            self._cls.append(np.full(len(b), c, dtype=int))
            self._tp.append(tp)

    def _arrays(self):
        if self._cache is None:
            if self._scores:
                self._cache = (
                    np.concatenate(self._scores),
                    np.concatenate(self._cls),
                    np.concatenate(self._tp),
                )
            else:
                self._cache = (np.zeros(0), np.zeros(0, dtype=int), np.zeros(0, dtype=bool))
        return self._cache

    @staticmethod
    def _prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        return p, r, f1

    def at_threshold(self, conf: float) -> dict:
        scores, cls, tp = self._arrays()
        keep = scores >= conf
        tp_c = np.bincount(cls[keep & tp], minlength=self.nc)
        fp_c = np.bincount(cls[keep & ~tp], minlength=self.nc)
        fn_c = self.n_gt - tp_c
        per_class = []
        for c in range(self.nc):
            p, r, f1 = self._prf(int(tp_c[c]), int(fp_c[c]), int(fn_c[c]))
            per_class.append(
                {"class_id": c, "precision": p, "recall": r, "f1": f1,
                 "tp": int(tp_c[c]), "fp": int(fp_c[c]), "fn": int(fn_c[c]),
                 "n_gt": int(self.n_gt[c])}
            )
        p, r, f1 = self._prf(int(tp_c.sum()), int(fp_c.sum()), int(fn_c.sum()))
        return {"conf": float(conf), "precision": p, "recall": r, "f1": f1,
                "tp": int(tp_c.sum()), "fp": int(fp_c.sum()), "fn": int(fn_c.sum()),
                "per_class": per_class}

    def average_precision(self) -> np.ndarray:
        """Per-class AP at the evaluator's IoU threshold (101-point interpolation, as in COCO)."""
        scores, cls, tp = self._arrays()
        ap = np.zeros(self.nc)
        recall_points = np.linspace(0, 1, 101)
        for c in range(self.nc):
            if self.n_gt[c] == 0:
                continue
            m = cls == c
            order = np.argsort(-scores[m], kind="stable")
            hits = tp[m][order]
            tpc = np.cumsum(hits)
            fpc = np.cumsum(~hits)
            recall = tpc / self.n_gt[c]
            precision = tpc / np.maximum(tpc + fpc, 1)
            # precision envelope, then sample it at fixed recall points
            envelope = np.maximum.accumulate(precision[::-1])[::-1] if len(precision) else precision
            idx = np.searchsorted(recall, recall_points, side="left")
            ap[c] = np.mean([envelope[i] if i < len(envelope) else 0.0 for i in idx])
        return ap

    def sweep(self, thresholds=DEFAULT_THRESHOLDS) -> list[dict]:
        return [self.at_threshold(t) for t in thresholds]

    def best(self, thresholds=DEFAULT_THRESHOLDS):
        """Returns (best_overall, best_per_class, all_rows)."""
        rows = self.sweep(thresholds)
        best = max(rows, key=lambda r: r["f1"])
        best_pc = []
        for c in range(self.nc):
            br = max(rows, key=lambda r: r["per_class"][c]["f1"])
            best_pc.append({"conf": br["conf"], **br["per_class"][c]})
        return best, best_pc, rows


# --------------------------------------------------------------------------- #
# Results logging
# --------------------------------------------------------------------------- #
def append_metrics_row(row: dict, path=None) -> Path:
    """Append one row to results/metrics.csv (created if missing)."""
    import pandas as pd

    path = Path(path) if path else PROJECT_ROOT / "results" / "metrics.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    new = pd.DataFrame([row])
    if path.exists() and path.stat().st_size > 0:
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new.to_csv(path, index=False)
    return path

In [ ]:
%%writefile scripts/fix_prepared_data.py
#!/usr/bin/env python
"""Fix an already-prepared data/{train,val,test}/{images,labels} copy of RDD2022.

Use this instead of convert_voc_to_yolo.py + make_split.py when the dataset was
downloaded pre-converted (e.g. from Kaggle). Two problems are fixed:

1. Class ids. The prepared labels use 5 ids: 0 D00, 1 D10, 2 D20, 3 "other
   corruption" (D43/D44/D50/Repair merged) and 4 D40 pothole. The per-class box
   totals (26016 / 11830 / 10617 / 6544 for ids 0/1/2/4) match the published RDD2022
   counts for D00 / D10 / D20 / D40. The project uses the 4 standard classes, so id 3
   boxes are dropped and id 4 becomes 3. This is only done when id 4 is present.

2. The split. The prepared split is random per image, so near-identical neighbouring
   frames land in train and val/test. Images are regrouped into blocks of consecutive
   file numbers per country (same heuristic as make_split.py) and whole blocks are
   assigned to one split.

In place (default): files are moved between the split folders, so no extra disk.
All original labels and the original split are first saved to
<root>/original_labels.zip. Running twice is refused unless --force is given.

    python scripts/fix_prepared_data.py --root data

To a new folder (--dst): the source is left untouched (e.g. a read-only Kaggle input).
Labels are written to <dst> and images are symlinked (copied if symlinks fail). If the
source was already fixed by this script, its split and classes are mirrored as they are.

    python scripts/fix_prepared_data.py --root /kaggle/input/rdd2022 --dst /kaggle/working/data
"""
from __future__ import annotations

import argparse
import os
import random
import re
import shutil
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

from tqdm import tqdm

from utils import IMG_EXTS, PROJECT_ROOT

SPLITS = ("train", "val", "test")
CLASS_REMAP = {0: 0, 1: 1, 2: 2, 4: 3}  # source id -> project id; anything else is dropped
MARKER = ".fixed_by_fix_prepared_data"


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--root", default="data", help="folder holding train/, val/, test/")
    p.add_argument("--dst", default=None, help="write the fixed dataset here instead of changing --root")
    p.add_argument("--val-frac", type=float, default=0.15)
    p.add_argument("--test-frac", type=float, default=0.15)
    p.add_argument("--block-size", type=int, default=50, help="consecutive images treated as one sequence")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--keep-classes", action="store_true", help="only re-split, do not remap class ids")
    p.add_argument("--force", action="store_true", help="run even if the marker file says it already ran")
    return p.parse_args()


def country_and_index(stem: str):
    m = re.match(r"(.+?)_(\d+)$", stem)
    return (m.group(1), int(m.group(2))) if m else (stem, None)


def remap_label_text(text: str, dropped: Counter) -> str:
    out = []
    for line in text.splitlines():
        fields = line.split()
        if not fields:
            continue
        cid = int(float(fields[0]))
        if cid not in CLASS_REMAP:
            dropped[cid] += 1
            continue
        out.append(" ".join([str(CLASS_REMAP[cid])] + fields[1:]))
    return "\n".join(out) + ("\n" if out else "")


def has_class_id(label_dirs, class_id: int) -> bool:
    prefix = f"{class_id} "
    for d in label_dirs:
        for f in d.glob("*.txt"):
            with open(f, encoding="utf-8") as fh:
                if any(line.startswith(prefix) for line in fh):
                    return True
    return False


def link_or_copy(src: Path, dst: Path, state: dict) -> None:
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if not state["copy"]:
        try:
            os.symlink(src.resolve(), dst)
            return
        except OSError:
            state["copy"] = True
            print("Symlinks are not available here; copying images instead.")
    shutil.copy2(src, dst)


def block_split(items, args) -> dict:
    """Map each (country, block) group to a split, stratified by country."""
    groups = defaultdict(list)
    for item in items:
        country, idx = country_and_index(item[0])
        groups[(country, idx // args.block_size if idx is not None else item[0])].append(item)
    rng = random.Random(args.seed)
    by_country = defaultdict(list)
    for key in groups:
        by_country[key[0]].append(key)
    target = {}
    for country, keys in sorted(by_country.items()):
        keys = sorted(keys, key=str)
        rng.shuffle(keys)
        n = len(keys)
        n_test = max(1, round(n * args.test_frac)) if n >= 3 else 0
        n_val = max(1, round(n * args.val_frac)) if n >= 3 else 0
        for i, key in enumerate(keys):
            target[key] = "test" if i < n_test else ("val" if i < n_test + n_val else "train")
    return groups, target


def main():
    args = parse_args()
    root = Path(args.root)
    if not root.is_absolute():
        root = PROJECT_ROOT / root
    dst = Path(args.dst) if args.dst else root
    if not dst.is_absolute():
        dst = PROJECT_ROOT / dst
    in_place = dst.resolve() == root.resolve()
    if (dst / MARKER).exists() and not args.force:
        sys.exit(f"{dst / MARKER} exists: this data was already fixed. Use --force to run again.")

    # ---- inventory ----
    items = []  # (stem, current_split, image_path)
    for s in SPLITS:
        img_dir = root / s / "images"
        if img_dir.is_dir():
            items += [(p.stem, s, p) for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    if not items:
        sys.exit(f"No images found under {root}/<split>/images")
    stems = [i[0] for i in items]
    if len(set(stems)) != len(stems):
        sys.exit("The same image name appears in more than one split; cannot re-split safely.")

    # splits.csv also counts: uploads to Kaggle can drop dotfiles such as the marker
    source_fixed = ((root / MARKER).exists() or (root / "splits.csv").exists()) and not in_place
    if source_fixed:
        print(f"{root} was already fixed by this script: mirroring its split and classes unchanged.")
        remap = False
    elif args.keep_classes:
        remap = False
    else:
        remap = has_class_id([root / s / "labels" for s in SPLITS if (root / s / "labels").is_dir()], 4)
        print("Class id 4 (D40 pothole) found: remapping to the 4 project classes." if remap else
              "No class id 4 found: labels already use the 4 project classes, keeping them.")

    # ---- backup (in place only; --dst never touches the source) ----
    if in_place:
        backup = root / "original_labels.zip"
        if not backup.exists():
            with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as zf:
                zf.writestr("original_split.csv", "stem,split\n" + "".join(f"{st},{sp}\n" for st, sp, _ in items))
                for stem, s, _ in tqdm(items, desc="Backing up labels"):
                    lbl = root / s / "labels" / f"{stem}.txt"
                    if lbl.exists():
                        zf.write(lbl, f"{s}/labels/{stem}.txt")
            print(f"Backed up original labels and split to {backup}")

    # ---- split ----
    if source_fixed:
        groups = {(stem, None): [(stem, s, img)] for stem, s, img in items}
        target = {key: members[0][1] for key, members in groups.items()}
    else:
        groups, target = block_split(items, args)

    for s in SPLITS:
        (dst / s / "images").mkdir(parents=True, exist_ok=True)
        (dst / s / "labels").mkdir(parents=True, exist_ok=True)

    dropped, moved, state = Counter(), 0, {"copy": False}
    class_counts = {s: Counter() for s in SPLITS}
    rows = []
    for key, members in tqdm(sorted(groups.items(), key=lambda kv: str(kv[0])), desc="Writing split"):
        dst_split = target[key]
        for stem, src_split, img in members:
            src_lbl = root / src_split / "labels" / f"{stem}.txt"
            text = src_lbl.read_text(encoding="utf-8") if src_lbl.exists() else ""
            if remap:
                text = remap_label_text(text, dropped)
            (dst / dst_split / "labels" / f"{stem}.txt").write_text(text, encoding="utf-8")
            if in_place:
                if src_split != dst_split:
                    if src_lbl.exists():
                        src_lbl.unlink()
                    os.replace(img, dst / dst_split / "images" / img.name)
                    moved += 1
            else:
                link_or_copy(img, dst / dst_split / "images" / img.name, state)
                moved += src_split != dst_split
            for line in text.splitlines():
                class_counts[dst_split][int(float(line.split()[0]))] += 1
            rows.append(f"{stem},{key[0]},{key[0]}_{key[1]},{dst_split}\n")

    (dst / "splits.csv").write_text("stem,country,group,split\n" + "".join(sorted(rows)), encoding="utf-8")
    (dst / MARKER).write_text(f"block_size={args.block_size} seed={args.seed} remap={remap}\n", encoding="utf-8")

    # ---- summary ----
    print(f"\n{moved} images changed split. Dropped boxes by source class: {dict(dropped)}")
    per_split = Counter(target[k] for k, m in groups.items() for _ in m)
    for s in SPLITS:
        print(f"{s:>5}: {per_split[s]:>6} images | boxes per class {dict(sorted(class_counts[s].items()))}")
    print(f"\nWrote {dst / 'splits.csv'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/train.py
#!/usr/bin/env python
"""Train a YOLO or RT-DETR model on the road damage dataset (Ultralytics).

Examples:
    # baseline
    python scripts/train.py --model yolov8s.pt --imgsz 640 --epochs 100

    # stronger model, larger images
    python scripts/train.py --model yolov8m.pt --imgsz 1024 --epochs 100 --batch 8

    # RT-DETR comparison
    python scripts/train.py --model rtdetr-l.pt --epochs 100 --batch 8

    # from scratch (no COCO pretraining): use a model yaml instead of weights
    python scripts/train.py --model yolov8s.yaml --name scratch_yolov8s

    # ablation: extra Ultralytics arguments with --set
    python scripts/train.py --model yolov8s.pt --set mixup=0.1 --set degrees=5 --name aug_mixup

    # hyperparameter evolution
    python scripts/train.py --model yolov8s.pt --tune --tune-iterations 50 --tune-epochs 30

    # resume an interrupted run
    python scripts/train.py --resume runs/detect/yolov8s_640_s42/weights/last.pt

Every run saves its settings to <run_dir>/cli_args.json for reproducibility.
Use the same --seed, --epochs and --imgsz when comparing models.
"""
from __future__ import annotations

import argparse
import ast
import os
import shutil
import time
from pathlib import Path

from utils import PROJECT_ROOT, load_model, resolve_data_yaml, save_json, set_seed


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", default="yolov8s.pt",
                   help="weights (.pt = pretrained) or a model yaml (e.g. yolov8s.yaml = from scratch)")
    p.add_argument("--data", default="configs/rdd2022.yaml")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--batch", type=int, default=16, help="-1 = auto batch size")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--device", default=None, help="e.g. 0, 0,1 or cpu (default: auto)")
    p.add_argument("--workers", type=int, default=8)
    p.add_argument("--patience", type=int, default=30, help="early-stopping patience (epochs)")
    p.add_argument("--optimizer", default="auto")
    p.add_argument("--lr0", type=float, default=None)
    p.add_argument("--cos-lr", action="store_true", help="cosine learning-rate schedule")
    p.add_argument("--close-mosaic", type=int, default=10, help="disable mosaic for the last N epochs")
    p.add_argument("--freeze", type=int, default=None, help="freeze the first N layers")
    p.add_argument("--cache", choices=["ram", "disk"], default=None, help="cache images for faster epochs")
    p.add_argument("--no-deterministic", action="store_true", help="faster, but not bit-reproducible")
    p.add_argument("--project", default="runs/detect")
    p.add_argument("--name", default=None, help="run name (default: <model>_<imgsz>_s<seed>)")
    p.add_argument("--resume", default=None, help="path to last.pt (or resume.pt) of an interrupted run")
    p.add_argument("--max-hours", type=float, default=None,
                   help="stop cleanly before this many hours (e.g. cloud session limits); "
                        "writes weights/resume.pt to continue with --resume")
    p.add_argument("--set", dest="extra", action="append", default=[], metavar="KEY=VALUE",
                   help="any extra Ultralytics argument, repeatable (e.g. --set mixup=0.1)")
    p.add_argument("--wandb", action="store_true", help="log to Weights & Biases (run `wandb login` first)")
    p.add_argument("--tune", action="store_true", help="run hyperparameter evolution instead of training")
    p.add_argument("--tune-iterations", type=int, default=50)
    p.add_argument("--tune-epochs", type=int, default=30)
    return p.parse_args()


def parse_extra(pairs):
    out = {}
    for item in pairs:
        if "=" not in item:
            raise SystemExit(f"--set expects KEY=VALUE, got '{item}'")
        key, value = item.split("=", 1)
        try:
            out[key] = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            out[key] = value
    return out


def configure_wandb(enabled: bool) -> None:
    try:
        from ultralytics import settings

        settings.update({"wandb": bool(enabled)})
    except Exception as e:  # settings API differs between versions
        print(f"Could not toggle the W&B setting: {e}")
    if enabled:
        os.environ.setdefault("WANDB_PROJECT", "road-damage-detection")


def add_run_callbacks(model, max_hours: float | None) -> None:
    """Optional time budget, plus PAUSED / resume.pt bookkeeping for interrupted runs.

    Ultralytics strips the optimizer from last.pt whenever training ends, so a run
    stopped early could not be resumed. Before stopping for the time budget, last.pt
    (still complete at that point) is copied to resume.pt and a PAUSED file is written.
    When a run finishes normally both are removed.
    """
    start = time.time()
    state = {"paused": False}

    def on_fit_epoch_end(trainer):
        if not max_hours or state["paused"] or trainer.stop:  # also called again by final_eval
            return
        elapsed = time.time() - start
        epochs_run = trainer.epoch + 1 - trainer.start_epoch
        if elapsed + 1.2 * elapsed / max(epochs_run, 1) > max_hours * 3600:
            shutil.copy2(trainer.last, trainer.wdir / "resume.pt")
            trainer.stop = state["paused"] = True
            print(f"\nTime budget of {max_hours} h reached after epoch {trainer.epoch + 1}/{trainer.epochs}. "
                  f"Continue with: --resume {trainer.wdir / 'resume.pt'}", flush=True)

    def on_train_end(trainer):
        paused = Path(trainer.save_dir) / "PAUSED"
        if state["paused"]:
            paused.write_text(f"stopped after epoch {trainer.epoch + 1}\n", encoding="utf-8")
        else:
            paused.unlink(missing_ok=True)
            (trainer.wdir / "resume.pt").unlink(missing_ok=True)

    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)
    model.add_callback("on_train_end", on_train_end)


def main():
    args = parse_args()
    set_seed(args.seed)
    configure_wandb(args.wandb)
    project = str((PROJECT_ROOT / args.project).resolve())

    if args.resume:
        resolve_data_yaml(args.data)  # a fresh (cloud) session needs the resolved yaml the run points to
        model = load_model(args.resume)
        add_run_callbacks(model, args.max_hours)
        model.train(resume=True)
        return

    data_yaml = str(resolve_data_yaml(args.data))
    name = args.name or f"{Path(args.model).stem}_{args.imgsz}_s{args.seed}"
    model = load_model(args.model)

    common = dict(data=data_yaml, imgsz=args.imgsz, device=args.device, project=project, name=name,
                  seed=args.seed, deterministic=not args.no_deterministic, workers=args.workers)
    extra = parse_extra(args.extra)

    if args.tune:
        print(f"Hyperparameter evolution: {args.tune_iterations} iterations x {args.tune_epochs} epochs")
        model.tune(epochs=args.tune_epochs, iterations=args.tune_iterations, optimizer="AdamW",
                   plots=False, save=False, val=False, **common, **extra)
        return

    kwargs = dict(common, epochs=args.epochs, batch=args.batch, patience=args.patience,
                  optimizer=args.optimizer, cos_lr=args.cos_lr, close_mosaic=args.close_mosaic,
                  lr0=args.lr0, freeze=args.freeze, cache=args.cache or False, **extra)
    kwargs = {k: v for k, v in kwargs.items() if v is not None}

    add_run_callbacks(model, args.max_hours)
    results = model.train(**kwargs)

    save_dir = Path(model.trainer.save_dir)
    try:
        import torch
        import ultralytics

        versions = {"ultralytics": ultralytics.__version__, "torch": torch.__version__}
    except Exception:
        versions = {}
    save_json({"cli_args": vars(args), "train_kwargs": kwargs, "versions": versions}, save_dir / "cli_args.json")

    print("\n" + "=" * 60)
    print(f"Run folder : {save_dir}")
    print(f"Best weights: {save_dir / 'weights' / 'best.pt'}")
    try:
        print(f"Val mAP@0.5 = {results.box.map50:.4f} | mAP@0.5:0.95 = {results.box.map:.4f}")
    except Exception:
        pass
    print(f"Next: python scripts/evaluate.py --weights {save_dir / 'weights' / 'best.pt'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/evaluate.py
#!/usr/bin/env python
"""Evaluate a trained model: F1 @ IoU 0.5, mAP, per-class metrics and speed.

Two evaluations are combined:
  1. Ultralytics validation: mAP@0.5, mAP@0.5:0.95, per-class AP, plots and confusion matrix.
  2. A custom detection evaluation: P / R / F1 at IoU 0.5 over a sweep of confidence
     thresholds (the RDD challenge metric), overall and per class.

Examples:
    python scripts/evaluate.py --weights runs/detect/yolov8s_640_s42/weights/best.pt

    # tune the threshold on val, then apply that exact threshold to the held-out test split
    python scripts/evaluate.py --weights best.pt --split val
    python scripts/evaluate.py --weights best.pt --split test --fixed-conf 0.30

The best-threshold F1 on the same split you tuned on is optimistic. For honest
numbers, pick the threshold on val and report the F1 at that --fixed-conf on test.

Outputs:
    results/evaluations/<run>_<split>.json   full results
    results/metrics.csv                      one row appended per evaluation

Speed numbers come from Ultralytics' per-image timing (preprocess + inference +
postprocess). Use export_model.py for a dedicated batch-size-1 latency benchmark.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
from tqdm import tqdm

from utils import (DEFAULT_THRESHOLDS, PROJECT_ROOT, DetectionEvaluator, append_metrics_row, get_split_dirs,
                   list_images, load_model, load_names, read_yolo_labels, resolve_data_yaml, resolve_weights,
                   save_json)


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--weights", required=True)
    p.add_argument("--data", default="configs/rdd2022.yaml")
    p.add_argument("--split", default="val", help="val or test")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--device", default=None)
    p.add_argument("--workers", type=int, default=8, help="dataloader processes for the validation pass")
    p.add_argument("--conf", type=float, default=0.001, help="minimum confidence kept for the sweep")
    p.add_argument("--nms-iou", type=float, default=0.6, help="NMS IoU threshold")
    p.add_argument("--fixed-conf", type=float, default=None,
                   help="also report F1 at this exact threshold (e.g. one tuned on val)")
    p.add_argument("--name", default=None, help="label for the results (default: run folder name)")
    p.add_argument("--max-images", type=int, default=None, help="evaluate on the first N images only (quick check)")
    p.add_argument("--skip-val", action="store_true", help="skip the Ultralytics validation (mAP, plots)")
    return p.parse_args()


def run_name(weights: str) -> str:
    p = Path(weights)
    return p.parent.parent.name if p.parent.name == "weights" else p.stem


def main():
    args = parse_args()
    weights = resolve_weights(args.weights)
    name = args.name or run_name(weights)
    data_yaml = resolve_data_yaml(args.data)
    names = load_names(data_yaml)
    nc = len(names)
    model = load_model(weights)

    # ---------- 1. Ultralytics validation (mAP, plots) ----------
    ul = {}
    if not args.skip_val:
        m = model.val(data=str(data_yaml), split=args.split, imgsz=args.imgsz, batch=args.batch,
                      conf=args.conf, iou=args.nms_iou, device=args.device, workers=args.workers, plots=True,
                      project=str(PROJECT_ROOT / "runs" / "val"), name=f"{name}_{args.split}",
                      exist_ok=True, verbose=False)
        ul = {"map50": float(m.box.map50), "map50_95": float(m.box.map),
              "precision": float(m.box.mp), "recall": float(m.box.mr),
              "per_class": {names[int(c)]: {"precision": float(m.box.p[i]), "recall": float(m.box.r[i]),
                                            "ap50": float(m.box.ap50[i]), "ap50_95": float(m.box.ap[i])}
                            for i, c in enumerate(m.box.ap_class_index)}}

    # ---------- 2. Custom F1 sweep + speed ----------
    img_dir, lbl_dir = get_split_dirs(data_yaml, args.split)
    images = list_images(img_dir)
    if args.max_images:
        images = images[:args.max_images]
    if not images:
        raise SystemExit(f"No images found in {img_dir}")

    evaluator = DetectionEvaluator(nc, iou_thr=0.5)
    timings = []
    stream = model.predict(source=[str(p) for p in images], imgsz=args.imgsz, conf=args.conf,
                           iou=args.nms_iou, device=args.device, batch=args.batch, stream=True, verbose=False)
    for r in tqdm(stream, total=len(images), desc=f"Predicting {args.split}"):
        gt_cls, gt_boxes = read_yolo_labels(lbl_dir / f"{Path(r.path).stem}.txt")
        b = r.boxes
        evaluator.add(b.xyxyn.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy(), gt_boxes, gt_cls)
        timings.append((r.speed["preprocess"], r.speed["inference"], r.speed["postprocess"]))

    best, best_pc, rows = evaluator.best(DEFAULT_THRESHOLDS)
    fixed = evaluator.at_threshold(args.fixed_conf) if args.fixed_conf is not None else None

    t = np.array(timings[5:] if len(timings) > 10 else timings)  # skip warm-up images
    ms_per_img = float(t.sum(axis=1).mean())
    fps = 1000.0 / ms_per_img if ms_per_img > 0 else 0.0
    try:
        params_m = sum(p.numel() for p in model.model.parameters()) / 1e6
    except Exception:
        params_m = None

    # ---------- report ----------
    print("\n" + "=" * 64)
    print(f"Model: {name} | split: {args.split} | images: {len(images)} | imgsz: {args.imgsz}")
    print("=" * 64)
    if ul:
        print(f"mAP@0.5 = {ul['map50']:.4f} | mAP@0.5:0.95 = {ul['map50_95']:.4f}")
    print(f"Best F1@IoU0.5 = {best['f1']:.4f} at conf {best['conf']:.2f} "
          f"(P {best['precision']:.3f}, R {best['recall']:.3f})")
    if fixed:
        print(f"F1 at fixed conf {fixed['conf']:.2f} = {fixed['f1']:.4f} "
              f"(P {fixed['precision']:.3f}, R {fixed['recall']:.3f})")
    print(f"Speed: {ms_per_img:.1f} ms/img ({fps:.1f} FPS)" + (f" | params: {params_m:.1f}M" if params_m else ""))
    print(f"\n{'class':<8}{'GT':>7}{'best conf':>11}{'P':>8}{'R':>8}{'F1':>8}{'AP50':>8}")
    for c in range(nc):
        pc = best_pc[c]
        ap50 = ul.get("per_class", {}).get(names[c], {}).get("ap50")
        print(f"{names[c]:<8}{pc['n_gt']:>7}{pc['conf']:>11.2f}{pc['precision']:>8.3f}{pc['recall']:>8.3f}"
              f"{pc['f1']:>8.3f}{(f'{ap50:.3f}' if ap50 is not None else '-'):>8}")

    # ---------- save ----------
    out = {"name": name, "weights": weights, "split": args.split, "imgsz": args.imgsz, "n_images": len(images),
           "ultralytics": ul, "best_f1": {k: v for k, v in best.items() if k != "per_class"},
           "best_per_class": {names[c]: best_pc[c] for c in range(nc)},
           "fixed_conf": ({k: v for k, v in fixed.items() if k != "per_class"} if fixed else None),
           "sweep": [{k: v for k, v in r.items() if k != "per_class"} for r in rows],
           "ms_per_img": ms_per_img, "fps": fps, "params_m": params_m}
    json_path = PROJECT_ROOT / "results" / "evaluations" / f"{name}_{args.split}.json"
    save_json(out, json_path)

    row = {"run": name, "split": args.split, "imgsz": args.imgsz, "params_M": params_m,
           "precision": best["precision"], "recall": best["recall"], "f1": best["f1"], "f1_conf": best["conf"],
           "map50": ul.get("map50"), "map50_95": ul.get("map50_95"), "ms_per_img": ms_per_img, "fps": fps}
    if fixed:
        row.update({"fixed_conf": fixed["conf"], "f1_at_fixed_conf": fixed["f1"]})
    for c in range(nc):
        row[f"ap50_{names[c]}"] = ul.get("per_class", {}).get(names[c], {}).get("ap50")
        row[f"f1_{names[c]}"] = best_pc[c]["f1"]
    csv_path = append_metrics_row(row)
    print(f"\nSaved {json_path}\nAppended a row to {csv_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile configs/rdd2022.yaml
# Prepared layout data/{train,val,test}/{images,labels}, fixed with scripts/fix_prepared_data.py
# (classes remapped to the 4 below, split by 50-frame blocks per country).
path: data/train
train: images
val: ../val/images
test: ../test/images

names:
  0: D00   # longitudinal crack
  1: D10   # transverse crack
  2: D20   # alligator crack
  3: D40   # pothole

## 4. Data

Finds the dataset, then writes a fixed copy (4 classes, leak-free block split) to the working folder. Images are symlinked, so this is fast and small.

In [ ]:
def find_dataset(roots, max_depth=5):
    """First folder below the roots that has train/images and val/images."""
    skip = {"images", "labels", "drive", "rdd", "outputs", "sample_data", ".config"}
    for root in map(Path, roots):
        if not root.is_dir():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            here = Path(dirpath)
            if (here / "train" / "images").is_dir() and (here / "val" / "images").is_dir():
                return here
            if len(here.relative_to(root).parts) >= max_depth:
                dirnames[:] = []
            else:
                dirnames[:] = [d for d in dirnames if d not in skip]
    return None


src = Path(DATA_DIR) if DATA_DIR else None
if src and src.suffix == ".zip":
    raw = WORK / "raw"
    if not raw.exists():
        print(f"Unzipping {src} ...")
        shutil.unpack_archive(str(src), str(raw))
    src = find_dataset([raw])
elif src is None:
    src = find_dataset(["/kaggle/input"] if ON_KAGGLE else ["/content"] if ON_COLAB else [Path.cwd()])
elif not (src / "train" / "images").is_dir():
    src = find_dataset([src])
if src is None:
    raise FileNotFoundError("Dataset not found. Add it as an input (Kaggle) or set DATA_DIR to the folder or .zip "
                            "that contains train/ val/ test/ with images/ and labels/.")
print("Dataset source:", src)

DATA = WORK / "data"
if not (DATA / ".fixed_by_fix_prepared_data").exists():
    run([sys.executable, "scripts/fix_prepared_data.py", "--root", src, "--dst", DATA])
else:
    print("Prepared data already in", DATA)

## 5. Train

Resumes automatically if a resumable checkpoint exists. Stops cleanly after `MAX_HOURS`; run the notebook again to continue.

In [ ]:
def resumable(pt):
    """True if the checkpoint still has its optimizer state (training can continue from it)."""
    if not Path(pt).is_file():
        return False
    ckpt = torch.load(pt, map_location="cpu", weights_only=False)
    return ckpt.get("optimizer") is not None and ckpt.get("epoch", -1) != -1


def bring_back_previous_run(run_dir):
    """Copy a previous session's run folder (from RESUME_FROM) into place, once."""
    if not RESUME_FROM or run_dir.exists():
        return
    matches = [Path(p) for p in glob.glob(f"{RESUME_FROM}/**/{RUN_NAME}", recursive=True) if Path(p).is_dir()]
    if Path(RESUME_FROM).name == RUN_NAME:
        matches.insert(0, Path(RESUME_FROM))
    if not matches:
        raise FileNotFoundError(f"No folder named {RUN_NAME} under RESUME_FROM={RESUME_FROM}")
    shutil.copytree(matches[0], run_dir)
    print(f"Copied previous run {matches[0]} -> {run_dir}")

In [ ]:
run_dir = RUNS / "detect" / RUN_NAME
weights_dir = run_dir / "weights"
bring_back_previous_run(run_dir)

if (run_dir / "DONE").exists():
    print("Training already finished:", run_dir)
else:
    ckpt = next((p for p in (weights_dir / "last.pt", weights_dir / "resume.pt") if resumable(p)), None)
    if ckpt:
        print("Resuming from", ckpt)
        cmd = [sys.executable, "scripts/train.py", "--resume", ckpt]
    else:
        cmd = [sys.executable, "scripts/train.py", "--model", WEIGHTS, "--imgsz", IMGSZ, "--epochs", EPOCHS,
               "--batch", BATCH, "--seed", SEED, "--device", DEVICE, "--workers", WORKERS,
               "--project", RUNS / "detect", "--name", RUN_NAME, "--set", "exist_ok=True"] + shlex.split(EXTRA_ARGS)
    if MAX_HOURS:
        cmd += ["--max-hours", MAX_HOURS]
    run(cmd)
    if (weights_dir / "best.pt").exists() and not (run_dir / "PAUSED").exists():
        (run_dir / "DONE").write_text(time.strftime("%Y-%m-%d %H:%M:%S\n"))

FINISHED = (run_dir / "DONE").exists()
print("Finished" if FINISHED else "PAUSED: run this notebook again (with RESUME_FROM on Kaggle) to continue.")

## 6. Training curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(run_dir / "results.csv")
df.columns = df.columns.str.strip()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.plot(x="epoch", y=[c for c in df.columns if c.startswith("train/")], ax=axes[0], title="Training loss")
df.plot(x="epoch", y=["metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/precision(B)", "metrics/recall(B)"],
        ax=axes[1], title="Validation metrics")
plt.tight_layout()
plt.show()
df.tail()

## 7. Evaluate on val and test

F1 @ IoU 0.5 (the RDD challenge metric) and mAP@0.5, overall and per class.

In [ ]:
# Pick the confidence threshold on val, then report test F1 at exactly that threshold (honest numbers).
if FINISHED:
    best = weights_dir / "best.pt"
    eval_device = DEVICE.split(",")[0]
    run([sys.executable, "scripts/evaluate.py", "--weights", best, "--split", "val", "--imgsz", IMGSZ,
         "--device", eval_device, "--workers", WORKERS, "--name", RUN_NAME])
    val = json.loads((WORK / "results" / "evaluations" / f"{RUN_NAME}_val.json").read_text())
    conf = val["best_f1"]["conf"]
    run([sys.executable, "scripts/evaluate.py", "--weights", best, "--split", "test", "--imgsz", IMGSZ,
         "--device", eval_device, "--workers", WORKERS, "--name", RUN_NAME, "--fixed-conf", conf])
else:
    print("Training not finished yet; skipping evaluation.")

## 8. Save outputs

In [ ]:
# Copy evaluation results next to the runs and zip the run folder for easy download.
import pandas as pd

if (WORK / "results").exists():
    shutil.copytree(WORK / "results", OUT / "results", dirs_exist_ok=True)
archive = shutil.make_archive(str(OUT / RUN_NAME), "zip", run_dir)
print("Run folder:", run_dir)
print("Zipped    :", archive)
metrics_csv = OUT / "results" / "metrics.csv"
if metrics_csv.exists():
    cols = ["run", "split", "precision", "recall", "f1", "f1_conf", "fixed_conf", "f1_at_fixed_conf",
            "map50", "map50_95", "fps"]
    table = pd.read_csv(metrics_csv)
    display(table[[c for c in cols if c in table.columns]])